In [3]:
%cd ..

/scratch/big/home/daawic/BSc-Thesis


In [4]:
import os
import torch
import matplotlib.pyplot as plt
from torchvision.transforms import transforms
from project.models import EDMCallum
from project.util.device import get_available_acc
from project.util.plotting import plot_sample
from project.util.data import ReplayMemoryData
from project.util.metrics import PSNR, MSE

In [6]:
PATH = os.path.join("..", "checkpoints", "diff", "BreakoutNoDiff20.pt")
DATA = os.path.join("..", "checkpoints", "memory", "Breakout.pt")

In [7]:
device = "cuda:1"

In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(2),
    transforms.Normalize(0.5, 0.5),
])

In [9]:
data = ReplayMemoryData(
    memory=DATA,
    transform=transform,
    cap=500_000,
    train=False,
)

In [10]:
model = EDMCallum.from_checkpoint(PATH, device, U=10).to(device)

In [11]:
n = 100

x = torch.zeros((n, 4, 88, 88), device=device)

for i, img in enumerate(torch.randperm(500_000)[:n]):
    x[i] = data[img].to(device)

mask = torch.ones_like(x, device=device)
mask[:, 2:] = 0

x_masked = x * mask
x_inpainted = model.inpaint(x, mask)

100%|██████████| 32/32 [07:02<00:00, 13.19s/it]


In [ ]:
mses = torch.zeros(n)

for i in range(n):
    mses[i] = MSE(x[i, 2:], x_inpainted[i, 2:])

In [ ]:
print(f"MSE: {mses.mean()} (error: {mses.std() / n ** 0.5})")

PSNR: 36.542118072509766 (error: 0.6264012455940247)
MSE: 0.0015025989850983024 (error: 0.00011421048839110881)
